# Setup

- macOS: Run 'brew install poppler'
- Ubuntu/Debian: Run 'apt-get install poppler-utils'
- Windows: Install from https://github.com/oschwartz10612/poppler-windows/releases/ and add to PATH

In [ ]:
# !pip install typhoon-ocr==0.4.1
# !pip install python-dotenv
# !pip install lxml
# !pip install html5lib
# !pip install beautifulsoup4

In [ ]:
from dotenv import load_dotenv
import os
import pandas as pd
from typhoon_ocr import ocr_document
from io import StringIO
import re
import time

import sys
import os

sys.path.append(os.path.abspath(".."))
from utils import *

load_dotenv()

In [ ]:
KEYWORD = "referendum"

REFERENDUM_LIST = ["เห็นชอบ", "ไม่เห็นชอบ", "ไม่แสดงความคิดเห็น"]

# OCR

In [ ]:
API_KEY = os.getenv("TYPHOON_OCR_API_KEY")

### Reader Function

In [ ]:
def read_referendum(file_name: str, page_num: int):
    print(f"Processing Referendum - File: {file_name} - Page: {page_num}-{page_num+1}")
    safe_path = get_safe_pdf_path(file_name)
    
    data_json_page_i_1 = ocr_document(
        api_key=API_KEY,
        pdf_or_image_path=safe_path,
        page_num=page_num,
        target_image_dim=1600,
    )
    
    data_json_page_i_2 = ocr_document(
        api_key=API_KEY,
        pdf_or_image_path=safe_path,
        page_num=page_num+1,
        target_image_dim=1600,
    )

    data_json_page_i_1 = thai_to_arabic(data_json_page_i_1)
    data_json_page_i_2 = thai_to_arabic(data_json_page_i_2)

    df_page_i_2 = pd.read_html(StringIO(data_json_page_i_2))[0]
    
    df = pd.concat([df_page_i_2])
    
    total_eligible = re.search(
        r'ผู้มีสิทธิออกเสียงประชามติ.*?(\d[\d,]*)\s*คน',
        data_json_page_i_1
    )
    
    total_voted = re.search(
        r'มาแสดงตน.*?(\d[\d,]*)\s*คน',
        data_json_page_i_1
    )

    summary = {
        'ผู้มีสิทธิ': parse_number(total_eligible.group(1)) if total_eligible else 0,
        'มาใช้สิทธิ': parse_number(total_voted.group(1)) if total_voted else 0,
    }

    df = df.iloc[:, [-1]]
    df.columns = ['คะแนน']
    df = df.dropna().reset_index(drop=True)

    df = df[df['คะแนน'].astype(str).str.contains(r'\d')]

    text = " ".join(df['คะแนน'].astype(str))

    nums = re.findall(r'(\d[\d,]*)\s*\(', text)
    nums = list(map(int, nums))

    # fallback 1: keyword anchor
    if len(nums) < len(REFERENDUM_LIST):
        patterns = [
            r'เห็นชอบ\D{0,10}?(\d[\d,]*)',
            r'ไม่เห็นชอบ\D{0,10}?(\d[\d,]*)',
            r'ไม่แสดงความคิดเห็น\D{0,10}?(\d[\d,]*)',
        ]
        nums = []
        for pat in patterns:
            m = re.search(pat, text)
            nums.append(int(m.group(1).replace(',', '')) if m else 0)

    # fallback 2: ตัวเลขล้วนๆ ไม่มี keyword เช่น "111 271 30 130"
    if sum(1 for n in nums if n == 0) >= len(REFERENDUM_LIST) - 1:
        raw_nums = re.findall(r'\b(\d+)\b', text)
        raw_nums = list(map(int, raw_nums))
        if len(raw_nums) >= len(REFERENDUM_LIST):
            nums = raw_nums[:len(REFERENDUM_LIST)]

    df = pd.DataFrame({
        'รายการ': REFERENDUM_LIST,
        'คะแนน': nums[:len(REFERENDUM_LIST)]
    })

    save_ocr_output(file_name, page_num, summary, df)

    return summary, df

### Reader

In [ ]:
for file_config in OCR_FILES:
    total_pages = file_config["pages"]
    if KEYWORD in file_config["path"]:
        for page_num in range(1, total_pages, 2):
            read_referendum(file_config["path"], page_num)

# Check OCR Result

In [ ]:
dfs = []

for file_config in OCR_FILES:
    total_pages = file_config["pages"]
    if KEYWORD in file_config["path"]:
        for page_num in range(1, total_pages, 2):
            df_dict = load_data(file_config["path"], page_num)
            dfs.append(df_dict)

In [ ]:
dfs[0]